# Capítulo 5 · Regresión logística

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_05/04_regresion_logistica.ipynb)

**Aprendizaje supervisado · Parte 4**

Vamos a predecir la especie de una flor de **Iris** a partir de sus medidas, como en la lectura de árboles. Aunque se llame regresión, `LogisticRegression` sirve para **clasificar**.

El flujo es: **cargar datos → separar → escalar → entrenar → predecir → evaluar**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

## 1. Datos y exploración

Usaremos las cuatro medidas de cada flor para predecir su especie: **setosa, versicolor o virginica**.

Si falta `data/iris.csv`, ejecuta `uv run python scripts/download_data.py` desde la raíz del repositorio.

In [ ]:
df = pd.read_csv(DATA_DIR / "iris.csv")
df.columns = ["largo_sepalo", "ancho_sepalo", "largo_petalo", "ancho_petalo", "especie"]
df["especie"] = df["especie"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
df.head()

In [ ]:
print("Filas y columnas:", df.shape)
print("Valores nulos:", df.isna().sum().sum())
print("Flores por especie:\n", df["especie"].value_counts())

In [ ]:
sns.scatterplot(data=df, x="largo_petalo", y="ancho_petalo", hue="especie", palette="Set2")
plt.title("¿Qué especies se separan con las medidas del pétalo?")
plt.show()

## 2. Separar train y test

Quitamos duplicados antes de separar para evitar que una misma fila aparezca en ambos conjuntos. Reservamos el **20 % para la evaluación final**; `stratify` conserva la proporción de especies.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
X = df.drop(columns="especie")
y = df["especie"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 3. Entrenar el modelo

`StandardScaler` pone las medidas en una escala comparable. El `Pipeline` une ese paso con el clasificador: al llamar a `fit`, ambos aprenden **solo con train**.

In [ ]:
modelo = Pipeline([
    ("escalador", StandardScaler()),
    ("logistica", LogisticRegression()),
])
modelo.fit(X_train, y_train)

## 4. Predecir y evaluar

`predict` devuelve la especie predicha. La **accuracy** es la proporción de aciertos en test; la matriz de confusión muestra qué especies se confunden.

El reporte de clasificación muestra **precision** (cuántas predicciones de una especie son correctas), **recall** (cuántas flores de esa especie detecta), **F1-score** (combina ambas) y **support** (cuántas flores reales hay de cada especie).


In [ ]:
predicciones = modelo.predict(X_test)
print(f"Accuracy en test: {accuracy_score(y_test, predicciones):.1%}")
print(classification_report(y_test, predicciones, zero_division=0))

matriz = confusion_matrix(y_test, predicciones, labels=modelo.classes_)

plt.figure(figsize=(6, 4))
sns.heatmap(
    matriz, annot=True, fmt="d", cmap="Blues",
    xticklabels=modelo.classes_, yticklabels=modelo.classes_,
)
plt.xlabel("Especie predicha")
plt.ylabel("Especie real")
plt.title("Matriz de confusión")
plt.tight_layout()
plt.show()

## 5. Ver probabilidades

`predict_proba` devuelve una probabilidad por especie. Las columnas siguen el orden de `classes_`; la especie con mayor probabilidad es la predicción.

In [ ]:
flores = X_test.head()
pd.DataFrame(
    modelo.predict_proba(flores), columns=modelo.classes_, index=flores.index
).round(3)